[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/05_%E4%BA%BA%E8%B3%87_%E5%93%A1%E5%B7%A5%E9%9B%A2%E8%81%B7%E9%A2%A8%E9%9A%AA%E9%A0%90%E8%AD%A6/%E5%93%A1%E5%B7%A5%E9%9B%A2%E8%81%B7%E9%A2%A8%E9%9A%AA%E9%A0%90%E8%AD%A6_%E5%88%86%E9%A1%9E%E7%AF%87.ipynb)

# 👥 範例 05：人資管理—員工離職風險預警與人才挽留 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「企業人資主管 (HRBP)、管理主管與組織人資分析師（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以 HR 實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、人資資料集介紹 (`hr_turnover.csv`)

本數據集記錄了企業歷年 1,800 位員工在職考評、工作負荷與離職狀態：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 人資業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **所屬部門** | 類別型 (`object`) | 研發工程、業務銷售、行銷企劃、客戶服務、財務人資 | 團隊工作型態與外部挖角熱度 |
| **年資_年** | 數值型 (`float64`) | 3.0 (年) | 服務年限（2~3 年常為轉職高峰期） |
| **月薪_TWD** | 數值型 (`int64`) | 65,000 (元) | 固定月薪水準 |
| **過去一年加班時數** | 數值型 (`int64`) | 320 (小時) | 衡量過勞與工作負擔之客觀指標 |
| **最近一次績效評等** | 類別型 (`object`) | 特優(A)、良好(B+)、符合預期(B)、需改善(C) | 年度考核評定等級 |
| **最近升遷距今年數** | 數值型 (`float64`) | 3.0 (年) | 晉升機會是否停滯 |
| **工作滿意度評分** | 數值型 (`int64`) | 2 (分，範圍 1~5) | 滿意度問卷自評分數 |
| **是否離職 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：半年內是否提出離職** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視部門離職率與工時薪資分佈。
3. **特徵關聯性與視覺化**：繪製加班時數與工作滿意度對離職率之影響長條圖。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸」與「隨機森林分類器」。
6. **測試集大考與留才處遇**：評估 Recall 與混淆矩陣，輸出模型即時預估。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入人資員工離職分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並載入繁體中文字型，確保人資部門分析圖表的中文標籤正常呈現。

---
💡 **AI 提問小抄：**

1. 📌 **人資數據的隱私與合規**
   > 「在分析員工離職數據時，常常包含敏感的考績與薪資。如果是在跨國企業，GDPR 或個人資料保護法對機器學習訓練資料有何去識別化（Anonymization）要求？」


---  
### 【步驟 1】：載入資料與人資結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 hr_turnover.csv 並檢視資料概況
df = pd.read_csv("hr_turnover.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 整體離職率基準 ---")
print(df["是否離職"].value_counts(normalize=True))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  檢查 1,800 筆員工記錄的欄位與離職率基準水準（約 38%）。

---
💡 **AI 提問小抄：**

1. 📌 **組織流動率基準**
   > 「在科技軟體業與客服中心，年離職率 30% 以上常見嗎？機器學習模型如何幫助人資團隊分辨『健康的新陳代謝』與『異常的人才失血』？」


---  
### 【步驟 2】：加班工時與滿意度對離職率之交叉圖表繪製

In [ ]:
# [步驟 2 程式碼] 繪製滿意度評分與部門的離職率分析圖
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sat_order = df.groupby("工作滿意度評分")["是否離職"].apply(lambda s: (s == "是").mean())
sns.barplot(x=sat_order.index, y=sat_order.values * 100, palette="Reds_r", ax=axes[0])
axes[0].set_title("工作滿意度評分 vs 離職率 (%)")
axes[0].set_xlabel("滿意度問卷評分 (1最不滿意 ~ 5最滿意)")
axes[0].set_ylabel("離職率 (%)")

dept_order = df.groupby("所屬部門")["是否離職"].apply(lambda s: (s == "是").mean()).sort_values(ascending=False)
sns.barplot(x=dept_order.index, y=dept_order.values * 100, palette="coolwarm", ax=axes[1])
axes[1].set_title("各部門平均離職率比較 (%)")
axes[1].set_xlabel("部門")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將問卷滿意度 1~5 分與 5 大部門分組計算離職比例，呈現強烈直觀的負相關關係。

---
💡 **AI 提問小抄：**

1. 📌 **問卷作答心態偏差**
   > 「在企業內部調查中，有些員工即使心中不滿也會勾選 3 分（中立防衛心態）。機器學習如何結合客觀特徵（如加班時數、未升遷年數）來校正問卷評分的主觀盲點？」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["是否離職"])
y = (df["是否離職"] == "是").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ["年資_年", "月薪_TWD", "過去一年加班時數", "最近升遷距今年數", "工作滿意度評分"]
cat_cols = ["所屬部門", "最近一次績效評等"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集：{X_train.shape[0]} 筆，測試集：{X_test.shape[0]} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將數據拆分並使用 `ColumnTransformer` 將工時、薪資標準化，將部門與考績獨熱編碼。

---
💡 **AI 提問小抄：**

1. 📌 **考績類別編碼考量**
   > 「考績評等（A/B+/B/C）有明確高低之分。為什麼這裡採用 `OneHotEncoder` 而非對映成 4/3/2/1 數字？兩種方式在機器學習模型訓練上有何優缺點？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (邏輯回歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證比對邏輯回歸與隨機森林
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(random_state=42))])
pipe_rf = Pipeline([("preprocessor", preprocessor), ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="roc_auc")
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="roc_auc")

print(f"邏輯回歸 5 折交叉驗證 平均 ROC-AUC: {scores_lr.mean():.4f} (±{scores_lr.std():.4f})")
print(f"隨機森林 5 折交叉驗證 平均 ROC-AUC: {scores_rf.mean():.4f} (±{scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  運用 5 折交叉驗證評估兩種模型在判斷離職風險上的綜合鑑別力 (ROC-AUC)。

---
💡 **AI 提問小抄：**

1. 📌 **特徵重要度與可解釋性**
   > 「向高階主管報告時，主管通常會問『到底是什麼原因導致離職率高？』。邏輯回歸的係數與隨機森林的 Feature Importance，哪一個更容易讓人資長接受與理解？」


---  
### 【步驟 5】：測試集大考與指標解讀 (混淆矩陣, Recall, ROC-AUC)

In [ ]:
# [步驟 5 程式碼] 在全新測試集進行評估並繪製混淆矩陣
from sklearn.metrics import classification_report, confusion_matrix

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)

print("=== 測試集最終評估報告 ===")
print(classification_report(y_test, y_pred, target_names=["留任(0)", "離職(1)"]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=["預測留任", "預測離職"], yticklabels=["實際留任", "實際離職"])
plt.title("人資員工離職預警混淆矩陣")
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  在 360 筆獨立測試集上計算 Precision、Recall 與混淆矩陣。

---
💡 **AI 提問小抄：**

1. 📌 **Recall (召回率) 優先策略**
   > 「在人才管理中，『漏抓一個會離職的關鍵架構師』代價極高。為什麼人資部門通常更看重 Recall（寧可多約訪幾位員工，也不要遺漏想走的人）？」


---  
### 【步驟 6】：在職員工即時評估 (Inference) 與留才面談處遇

In [ ]:
# [步驟 6 程式碼] 保存模型管線並模擬在職員工離職風險即時推論
joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬一位剛填完季問卷的資深員工
employee = pd.DataFrame([{
    "所屬部門": "研發工程",
    "年資_年": 3.5,
    "月薪_TWD": 72000,
    "過去一年加班時數": 360,
    "最近一次績效評等": "特優(A)",
    "最近升遷距今年數": 3.5,
    "工作滿意度評分": 2
}])

loaded_model = joblib.load("model.joblib")
prob = loaded_model.predict_proba(employee)[0][1]
print(f"\n🔔 員工個人離職風險推估：{prob*100:.1f}%")
if prob >= 0.6:
    print("⚠️ 警訊：該同仁考績特優但工時過長且久未晉升，離職風險偏高！建議人資主管於兩週內安排 1 on 1 關懷面談。")
else:
    print("🟢 狀況穩定：目前無立即流失風險，維持常態性管理。")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  保存模型管線至 `model.joblib`，並示範即時輸入員工資料取得預警風險與面談指引。

---
💡 **AI 提問小抄：**

1. 📌 **組織內部推廣與信任建立**
   > 「如果讓員工知道公司用機器學習在預測誰會離職，可能引發焦慮或不信任。人資長該如何包裝這項技術為『主動關懷與員工體驗 (EX) 優化』專案？」
